# 03 – Train U-Net (Kaggle GPU / Colab GPU)

Trains the residual U-Net for shadow removal on preprocessed SD7K-512.

**Kaggle**: Attach `sd7k-512` Kaggle Dataset, then hit **Save & Run All**.
**Colab**: Mount Drive or clone repo, set paths, then run.

Folder layout expected inside `sd7k-512`:
```
train/input/    train/target/
val/input/      val/target/
test/input/     test/target/
```

In [ ]:
!pip -q install segmentation-models-pytorch albumentations pytorch-msssim pyyaml tqdm
import subprocess, sys
if 'google.colab' in sys.modules:
    subprocess.run(['apt-get', '-qq', 'install', 'tesseract-ocr'], check=True)

In [ ]:
import os, sys
from pathlib import Path
import torch

KAGGLE = os.path.exists('/kaggle')
COLAB  = 'google.colab' in sys.modules

if KAGGLE:
    SD7K_512 = '/kaggle/input/sd7k-512'  # <- your attached dataset path
    CKPT_DIR = '/kaggle/working/checkpoints'
    LOG_DIR  = '/kaggle/working/logs'
    PROJECT_ROOT = '/kaggle/working/DocRescue'
    # Clone repo or copy if uploaded as dataset
    if not Path(PROJECT_ROOT).exists():
        if Path('/kaggle/input/docrescue').exists():
            import shutil
            shutil.copytree('/kaggle/input/docrescue', PROJECT_ROOT)
        else:
            !git clone https://github.com/YOUR_USERNAME/DocRescue {PROJECT_ROOT}
    sys.path.insert(0, PROJECT_ROOT)
elif COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.exists('/content/drive/MyDrive/DocRescue'):
        PROJECT_ROOT = '/content/drive/MyDrive/DocRescue'
    else:
        PROJECT_ROOT = '/content/DocRescue'
        if not os.path.exists(PROJECT_ROOT):
            !git clone https://github.com/YOUR_USERNAME/DocRescue {PROJECT_ROOT}
    SD7K_512 = f'{PROJECT_ROOT}/data/sd7k_512'
    CKPT_DIR = f'{PROJECT_ROOT}/experiments/checkpoints'
    LOG_DIR  = f'{PROJECT_ROOT}/experiments/logs'
    sys.path.insert(0, PROJECT_ROOT)
else:
    PROJECT_ROOT = '.'
    SD7K_512 = 'data/sd7k_512'
    CKPT_DIR = 'experiments/checkpoints'
    LOG_DIR  = 'experiments/logs'
    sys.path.insert(0, '.')

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'Project root: {PROJECT_ROOT}')
print(f'SD7K-512: {SD7K_512}')

In [ ]:
# ── Build config dict (override paths for notebook) ────────────────────────
import yaml

config_file = os.path.join(PROJECT_ROOT, 'configs/config.yaml')
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['data']['sd7k_root'] = SD7K_512
cfg['training']['checkpoint_dir'] = CKPT_DIR
cfg['training']['log_dir'] = LOG_DIR

# Tune for your GPU VRAM:
#   T4  (16 GB) → batch 12-16
#   P100 (16 GB) → batch 12-16
#   V100 (32 GB) → batch 24
cfg['training']['batch_size'] = 12
cfg['training']['epochs'] = 50

print(yaml.dump(cfg))

In [ ]:
# ── Run training ────────────────────────────────────────────────────────────
from src.train import train
train(cfg, resume_ckpt=None)  # pass a ckpt path to resume

In [ ]:
# ── TensorBoard ──────────────────────────────────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}

In [ ]:
# ── Quick visual check of best checkpoint ──────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from src.models import build_model
from src.infer import load_checkpoint, gain_map_rescue
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(cfg).to(device)
load_checkpoint(model, f'{CKPT_DIR}/best.pth', device=device)

test_input_dir = os.path.join(SD7K_512, 'test', 'input')
test_target_dir = os.path.join(SD7K_512, 'test', 'target')
test_files = sorted(os.listdir(test_input_dir))[:3]

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for row, fn in enumerate(test_files):
    inp  = np.array(Image.open(os.path.join(test_input_dir, fn)).convert('RGB'))
    tgt  = np.array(Image.open(os.path.join(test_target_dir, fn)).convert('RGB'))
    pred = gain_map_rescue(inp, model, device=device)
    axes[row, 0].imshow(inp);  axes[row, 0].set_title('Input');        axes[row, 0].axis('off')
    axes[row, 1].imshow(pred); axes[row, 1].set_title('Predicted');    axes[row, 1].axis('off')
    axes[row, 2].imshow(tgt);  axes[row, 2].set_title('Target (GT)'); axes[row, 2].axis('off')

plt.suptitle('U-Net output (best checkpoint)', fontsize=13)
plt.tight_layout(); plt.show()